# RGB Barcode Decode Analysis: Native-2x vs SAA vs SAA+IBP

SR is performed on the **red Bayer channel** (RGGB → even rows/cols).
Each SR output is 1536 × 2048 (= full sensor resolution, 2× the red-channel LR size).

Two barcode sheets are included:
- **2/3/5 mil** — `2_3_5_mil_color_tilt 0.28256_settle50ms`
- **4/6 mil** — `4_6_mil_color_tilt 0.28256_settle50ms`

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import zxingcpp

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

RESULTS_DIR   = os.path.join(os.path.dirname(os.path.abspath("analysis.ipynb")), "results")
METHOD_NAMES  = ["Native-2x", "SAA", "SAA+IBP"]
COLORS_METHOD = {"Native-2x": "C0", "SAA": "C2", "SAA+IBP": "C3"}
NAME_MAP      = {"Native-2x": "native_2x", "SAA": "SAA", "SAA+IBP": "SAA_IBP"}
REP           = "rep00"   # which repeat to analyse

# Discover sessions from the results folder
sessions = sorted([
    d for d in os.listdir(RESULTS_DIR)
    if os.path.isdir(os.path.join(RESULTS_DIR, d))
    and not d.startswith('.')
])
print("Sessions found in results/:")
for s in sessions:
    reps = sorted(os.listdir(os.path.join(RESULTS_DIR, s)))
    print(f"  {s}  ->  {reps}")

## ROI Definitions

- `roi` — `(row_start, row_end, col_start, col_end)` in **HR pixel coordinates**
  (HR = 1536 × 2048)
- `pitch_mil` — barcode bar pitch in mils

Use the viewer cell below to locate coordinates (displayed at 1/4 scale → multiply by 4).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EDIT THIS CELL — fill in ROIs and pitches for every session
# ══════════════════════════════════════════════════════════════════════════════

SESSION_ROIS = {
    "2_3_5_mil_color_tilt 0.28256_settle50ms": [
        {"label": "2 mil", "roi": (225*4, 315*4, 110*4, 230*4), "pitch_mil": 2},
        {"label": "3 mil", "roi": (225*4, 315*4, 260*4, 430*4), "pitch_mil": 3},
        {"label": "5 mil", "roi": (100*4, 200*4, 120*4, 390*4), "pitch_mil": 5},
    ],
    "4_6_mil_color_tilt 0.28256_settle50ms": [
        {"label": "4 mil", "roi": (200*4, 300*4, 110*4, 340*4), "pitch_mil": 4},
        {"label": "6 mil", "roi": (100*4, 190*4, 80*4, 410*4), "pitch_mil": 6},
    ],
}

for session in sessions:
    assert session in SESSION_ROIS, f"Missing SESSION_ROIS entry for: {session}"
    for bc in SESSION_ROIS[session]:
        assert None not in bc["roi"],       f"Fill in ROI for {session} / {bc['label']}"
        assert bc["pitch_mil"] is not None, f"Fill in pitch_mil for {session} / {bc['label']}"

print("All ROIs defined:")
for session, barcodes in SESSION_ROIS.items():
    if session in sessions:
        print(f"  {session}  ->  {', '.join(str(bc['pitch_mil'])+' mil' for bc in barcodes)}")

## Image Viewer — Locate Barcode Coordinates

Displayed at 1/4 scale. Multiply any coordinate by 4 to get HR pixel coordinates.

In [ ]:
DS = 4  # display downscale (multiply imshow coords × DS to get HR pixels)

fig, axes = plt.subplots(1, len(sessions), figsize=(6 * len(sessions), 5), squeeze=False)

for ax, session in zip(axes[0], sessions):
    img_path = os.path.join(RESULTS_DIR, session, REP, 'native_2x.png')
    img = np.array(Image.open(img_path))
    ax.imshow(img[::DS, ::DS], cmap='gray', interpolation='nearest')
    ax.set_title(session.split('_color_')[0], fontsize=9)
    ax.set_xlabel(f'col//{DS}  (x {DS} -> HR px)')
    ax.set_ylabel(f'row//{DS}  (x {DS} -> HR px)')

plt.suptitle(f'Native-2x (red channel) at 1/{DS} scale — {REP}', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ROI overlay — draw defined barcodes on top of each session image
ROI_COLORS = ["red", "lime", "cyan", "orange", "magenta"]

fig, axes = plt.subplots(1, len(sessions), figsize=(6 * len(sessions), 5), squeeze=False)

for ax, session in zip(axes[0], sessions):
    img_path = os.path.join(RESULTS_DIR, session, REP, 'native_2x.png')
    img = np.array(Image.open(img_path))
    ax.imshow(img[::DS, ::DS], cmap='gray', interpolation='nearest')

    for i, bc in enumerate(SESSION_ROIS[session]):
        r0, r1, c0, c1 = bc["roi"]
        rect = patches.Rectangle(
            (c0 / DS, r0 / DS), (c1 - c0) / DS, (r1 - r0) / DS,
            linewidth=1.5, edgecolor=ROI_COLORS[i % len(ROI_COLORS)],
            facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(c0 / DS, r0 / DS - 3, bc["label"],
                color=ROI_COLORS[i % len(ROI_COLORS)], fontsize=7, va='bottom')

    ax.set_title(session.split('_color_')[0], fontsize=9)
    ax.set_xlabel(f'col//{DS}')
    ax.set_ylabel(f'row//{DS}')

plt.suptitle('ROI overlay (Native-2x)', fontsize=10)
plt.tight_layout()
plt.show()

## Decode Function

In [ ]:
N_TRIALS   = 25
MAX_JITTER = 2
RNG        = np.random.default_rng(42)


def decode_confidence(img_gray, roi, n_trials=N_TRIALS, max_jitter=MAX_JITTER):
    r0, r1, c0, c1 = roi
    H, W = img_gray.shape

    centre_res = zxingcpp.read_barcodes(img_gray[r0:r1, c0:c1])
    text       = centre_res[0].text if centre_res else None

    successes = 0
    for _ in range(n_trials):
        dr = int(RNG.integers(-max_jitter, max_jitter + 1))
        dc = int(RNG.integers(-max_jitter, max_jitter + 1))
        rr0 = max(0, r0 + dr);  rr1 = min(H, r1 + dr)
        rc0 = max(0, c0 + dc);  rc1 = min(W, c1 + dc)
        crop = img_gray[rr0:rr1, rc0:rc1]
        if crop.size > 0 and zxingcpp.read_barcodes(crop):
            successes += 1

    return text, successes / n_trials


def crop_roi(img_gray, roi):
    r0, r1, c0, c1 = roi
    return img_gray[r0:r1, c0:c1]


print(f"decode_confidence ready  (N_TRIALS={N_TRIALS}, max_jitter={MAX_JITTER} px)")

## Run Decoding

In [ ]:
all_results = {m: [] for m in METHOD_NAMES}

for session in sessions:
    rep_dir  = os.path.join(RESULTS_DIR, session, REP)
    barcodes = SESSION_ROIS[session]
    imgs     = {m: np.array(Image.open(os.path.join(rep_dir, f'{NAME_MAP[m]}.png')))
                for m in METHOD_NAMES}

    print(f"\n{'='*60}")
    print(f"  {session} / {REP}")
    print(f"{'='*60}")

    for method in METHOD_NAMES:
        print(f"\n  [{method}]")
        for bc in barcodes:
            text, conf = decode_confidence(imgs[method], bc["roi"])
            all_results[method].append({
                "session":   session,
                "label":     bc["label"],
                "pitch_mil": bc["pitch_mil"],
                "text":      text,
                "confidence": conf,
            })
            status = f"'{text}'" if text else "FAIL"
            print(f"    {bc['label']:8s}  {bc['pitch_mil']} mil  "
                  f"conf={conf:.2f}  -> {status}")

## Visual Comparison: Cropped Barcodes per Session

In [ ]:
for session in sessions:
    rep_dir  = os.path.join(RESULTS_DIR, session, REP)
    barcodes = SESSION_ROIS[session]
    imgs     = {m: np.array(Image.open(os.path.join(rep_dir, f'{NAME_MAP[m]}.png')))
                for m in METHOD_NAMES}
    session_res = {m: [r for r in all_results[m] if r["session"] == session]
                   for m in METHOD_NAMES}

    n_bc, n_met = len(barcodes), len(METHOD_NAMES)
    fig, axes = plt.subplots(n_bc, n_met, figsize=(5 * n_met, 3 * n_bc), squeeze=False)

    for row, bc in enumerate(barcodes):
        for col, method in enumerate(METHOD_NAMES):
            ax   = axes[row][col]
            r0, r1, c0, c1 = bc["roi"]
            ax.imshow(imgs[method][r0:r1, c0:c1], cmap="gray", interpolation="nearest")
            res  = next(r for r in session_res[method] if r["pitch_mil"] == bc["pitch_mil"])
            ax.set_title(f"{method}\nconf={res['confidence']:.2f}  '{res['text'] or '---'}'",
                         fontsize=8)
            ax.axis("off")
        axes[row][0].set_ylabel(f"{bc['label']}\n{bc['pitch_mil']} mil",
                                fontsize=9, rotation=0, ha="right", va="center", labelpad=5)

    plt.suptitle(session.split('_color_')[0], fontsize=9, y=1.01)
    plt.tight_layout()
    plt.show()

## Decode Confidence vs Barcode Pitch

In [ ]:
PIXEL_PITCH_UM  = 3.45
RED_LR_PITCH_UM = PIXEL_PITCH_UM * 2   # red-channel LR pixel pitch (6.9 um)
MIL_TO_UM       = 25.4

markers_all = {"Native-2x": "o", "SAA": "s", "SAA+IBP": "^"}

fig, ax = plt.subplots(figsize=(8, 5))

for method in METHOD_NAMES:
    pitches = [r["pitch_mil"] for r in all_results[method]]
    confs   = [r["confidence"] for r in all_results[method]]
    ax.plot(pitches, confs,
            color=COLORS_METHOD[method], marker=markers_all[method],
            markersize=8, linewidth=1.8, label=method, linestyle='-')

    for r in all_results[method]:
        ax.annotate(f"'{r['text']}'" if r['text'] else "x",
                    (r["pitch_mil"], r["confidence"]),
                    textcoords="offset points", xytext=(4, 4),
                    fontsize=6, color=COLORS_METHOD[method], alpha=0.7)

# Red-channel LR Nyquist (2 red-LR pixels per period)
nyquist_lr_mil = (RED_LR_PITCH_UM * 2) / MIL_TO_UM
ax.axvline(nyquist_lr_mil, color='gray', linestyle='--', alpha=0.5,
           label=f'Red-LR Nyquist ({nyquist_lr_mil:.2f} mil)')

# Sensor Nyquist (2 sensor pixels per period)
nyquist_sensor_mil = (PIXEL_PITCH_UM * 2) / MIL_TO_UM
ax.axvline(nyquist_sensor_mil, color='lightgray', linestyle=':', alpha=0.7,
           label=f'Sensor Nyquist ({nyquist_sensor_mil:.2f} mil)')

ax2 = ax.twiny()
ax2.set_xlim(np.array(ax.get_xlim()) * MIL_TO_UM)
ax2.set_xlabel("Barcode pitch (um)", fontsize=10)

all_pitches = sorted(set(r["pitch_mil"] for m in METHOD_NAMES for r in all_results[m]))
ax.set_xticks(all_pitches)
ax.set_xlabel("Barcode pitch (mil)", fontsize=11)
ax.set_ylabel("Decode confidence\n(fraction of jittered crops decoded)", fontsize=10)
ax.set_title("RGB (red channel) barcode decode confidence vs pitch", fontsize=11)
ax.set_ylim(-0.05, 1.10)
ax.set_xlim(left=0)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.2))
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.show()